# Hardseal-Occam — ARC Prize 2026 ARC-AGI-3 submission

**Bundle:** vendored as Kaggle Dataset `ricoallen/hardseal-occam-vendored` (mounted at `/kaggle/input/hardseal-occam-vendored/`).
- `occam_upstream/` — Sean Donahoe's [Occam](https://github.com/g-baskin/occam) ARC-AGI-3 solver (MIT, vendored verbatim at commit `e3be26a`)
- `hardseal_trace/` — `hardseal-trace` v0.2.0 cryptographic reasoning-trace primitive (Apache-2.0, vendored)
- `hardseal_occam/` — wrapper bridging Occam's `EventEmitter` to `SealedReasoningTracer` (CC0-1.0 OR MIT, this repo)

**Score parity vs stock Occam:** +0.0000% delta on local public-eval 25-game run (60.0132% wrapper vs 60.0132% stock to four decimals); 25/25 per-game sealed-trace chains validate via `verify_chain`. Wrapper is provably non-interfering.

Per ARC Prize 2026 contest rules, all wrapper code is `CC0-1.0 OR MIT`. See `NOTICE.md` and `LICENSES/` in the dataset for the full provenance map.

Source repo (mirror): https://github.com/ricojallen37-sketch/hardseal-occam

## Cell 1 — Install arc-agi SDK + set up vendored paths

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'arc-agi==0.9.8'])

In [ ]:
import os
import sys
from pathlib import Path

# Force ARC-AGI-3 SDK into competition mode (Kaggle infra also forces this,
# but explicit-is-better-than-implicit).
os.environ.setdefault('OPERATION_MODE', 'COMPETITION')

# Vendored code mounted from the Kaggle Dataset at /kaggle/input/hardseal-occam-vendored/
DATASET_ROOT = Path('/kaggle/input/hardseal-occam-vendored')
if not DATASET_ROOT.exists():
    # Local-run fallback: load from the contest repo dir alongside this notebook
    DATASET_ROOT = Path(__file__).resolve().parent if '__file__' in dir() else Path('.').resolve()

# Add vendored paths to sys.path so imports resolve
for p in [str(DATASET_ROOT), str(DATASET_ROOT / 'occam_upstream')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'DATASET_ROOT={DATASET_ROOT}')
print(f'Python {sys.version.split()[0]}')
print(f'OPERATION_MODE={os.environ.get("OPERATION_MODE")}')
print(f'sys.path[:3]: {sys.path[:3]}')

## Cell 2 — Provenance fingerprint (license map + content SHAs)

In [ ]:
import hashlib

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

fingerprint_files = [
    'occam_upstream/solver/orchestrator.py',
    'occam_upstream/solver/benchmark.py',
    'hardseal_trace/__init__.py',
    'hardseal_occam/tracer_callback.py',
    'hardseal_occam/runner.py',
    'LICENSE',
    'NOTICE.md',
    'LICENSES/MIT-occam.txt',
    'LICENSES/Apache-2.0-hardseal-trace.txt',
]
fingerprints = {}
for k in fingerprint_files:
    p = DATASET_ROOT / k
    fingerprints[k] = sha256(p) if p.exists() else 'MISSING'

for k, v in fingerprints.items():
    print(f'  {v[:16]:16s}  {k}')

## Cell 3 — Run the wrapper benchmark on Kaggle's eval environment

In [ ]:
import asyncio
import json
import time
import uuid

from solver.benchmark import BenchmarkRunner
from hardseal_occam.tracer_callback import HardsealOccamCallback
from hardseal_trace import verify_chain

RUN_UUID = str(uuid.uuid4())
HMAC_KEY = b'hardseal-occam-v0.4-hmac-key'
MAX_ACTIONS = 500000  # matches occam viewer/cli.py default; ample for eval

callback = HardsealOccamCallback(hmac_key=HMAC_KEY, run_uuid=RUN_UUID)
runner = BenchmarkRunner(
    event_callback=callback,
    mode='full',
    max_actions=MAX_ACTIONS,
)

t0 = time.time()
result = asyncio.run(runner.run())
wall_s = time.time() - t0

print(f'\nHardseal-Occam Kaggle Run')
print(f'  run_uuid:        {RUN_UUID}')
print(f'  RHAE:            {result["mean_rhae_pct"]:.4f}%')
print(f'  Games solved:    {result["games_solved"]}/{result["n_games"]}')
print(f'  Wall:            {wall_s:.1f}s')
print(f'  Scorecard:       {result.get("scorecard_id")}')

## Cell 4 — Verify the sealed-trace chain (per-game)

In [ ]:
per_game_chains = callback.export_per_game_chains()
validations = []
for i, gc in enumerate(per_game_chains):
    if not gc:
        validations.append((i, True, 'empty-chain'))
        continue
    try:
        ok, msg = verify_chain(gc, HMAC_KEY)
    except Exception as e:
        ok, msg = False, f'exception: {e}'
    validations.append((i, ok, msg))

all_valid = all(v[1] for v in validations)
total_traces = sum(len(gc) for gc in per_game_chains)
avg_pods = callback.average_pods()

print(f'per-game chains: {len(per_game_chains)} ({sum(1 for v in validations if v[1])}/{len(per_game_chains)} valid)')
print(f'total traces:    {total_traces}')
print(f'avg PODS:        {avg_pods}')
print(f'all valid:       {all_valid}')
for i, ok, msg in validations[:5]:
    print(f'  game[{i}]: ({ok}, {msg})')
if len(validations) > 5:
    print(f'  ... +{len(validations) - 5} more')

## Cell 5 — Persist sealed-trace JSONL + summary as Kaggle output artifacts

In [ ]:
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else DATASET_ROOT / 'kaggle_run'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sealed traces (full chain across all games, flat)
flat_chain = callback.export_chain()
chain_path = OUT_DIR / f'sealed_traces_{RUN_UUID}.jsonl'
with chain_path.open('w') as f:
    for trace in flat_chain:
        f.write(json.dumps(trace) + '\n')

summary = {
    'run_uuid': RUN_UUID,
    'agent': 'HardsealOccam',
    'agent_version': '0.4.0',
    'wrapped_solver': 'occam-solver@e3be26a (MIT)',
    'trace_primitive': 'hardseal-trace@0.2.0 (Apache-2.0)',
    'wall_s': round(wall_s, 1),
    'rhae_pct': result['mean_rhae_pct'],
    'games_solved': result['games_solved'],
    'n_games': result['n_games'],
    'scorecard_id': result.get('scorecard_id'),
    'trace_count': len(flat_chain),
    'avg_pods': avg_pods,
    'all_chains_valid': all_valid,
    'per_game_validation': [
        {'game_idx': i, 'valid': ok, 'msg': msg, 'trace_count': len(per_game_chains[i])}
        for i, ok, msg in validations
    ],
    'callback_stats': callback.stats(),
    'fingerprints': fingerprints,
}
summary_path = OUT_DIR / f'summary_{RUN_UUID}.json'
summary_path.write_text(json.dumps(summary, indent=2))

print(f'\nArtifacts written:')
print(f'  sealed traces: {chain_path} ({len(flat_chain)} traces)')
print(f'  summary:       {summary_path}')
print(f'\nFinal RHAE: {result["mean_rhae_pct"]:.4f}%  ({result["games_solved"]}/{result["n_games"]} games)')